# 321. Create Maximum Number

## Topic Alignment
- **Role Relevance**: Combines greedy selection with optimal merging, similar to feature selection from multiple sources, combining ranked lists in search/recommendation systems, and multi-source data fusion.
- **Scenario**: Applicable to selecting top-k items from multiple datasets while maintaining relative order, merging prediction results from multiple models, and constructing optimal sequences from distributed data sources.

## Metadata Summary
- Source: [LeetCode - Create Maximum Number](https://leetcode.com/problems/create-maximum-number/)
- Tags: `Greedy`, `Stack`, `Monotonic Stack`, `Array`
- Difficulty: Hard
- Recommended Priority: High

## Problem Statement
You are given two integer arrays `nums1` and `nums2` of lengths `m` and `n` respectively. `nums1` and `nums2` represent the digits of two numbers. You are also given an integer `k`.

Create the maximum number of length `k <= m + n` from digits of the two arrays. The relative order of the digits from the same array must be preserved.

Return an array of the `k` digits representing the answer.

**Constraints:**
- `m == nums1.length`
- `n == nums2.length`
- `1 <= m, n <= 500`
- `0 <= nums1[i], nums2[i] <= 9`
- `1 <= k <= m + n`

## Progressive Hints
- Hint 1: Break the problem into subproblems: select i digits from nums1 and k-i digits from nums2, then merge.
- Hint 2: For selecting the maximum subsequence of length k from a single array while preserving order, use a monotonic stack.
- Hint 3: When merging two sequences, always pick the lexicographically larger remaining sequence at each step.
- Hint 4: Try all possible splits: i from max(0, k-n) to min(k, m), and take the maximum result.
- Hint 5: Comparison during merge must be done carefully: compare entire remaining subsequences, not just current elements.

## Solution Overview
The solution involves three key steps:

1. **Extract maximum subsequence** from a single array:
   - Use a monotonic decreasing stack
   - For each element, pop smaller elements if we can afford to (have enough remaining elements)
   - This gives us the lexicographically maximum subsequence of length k

2. **Merge two subsequences** optimally:
   - At each position, choose from the sequence with lexicographically larger remaining part
   - Not just compare current elements, but entire suffixes

3. **Try all possible splits**:
   - For each valid split (i digits from nums1, k-i from nums2)
   - Extract, merge, and keep the maximum result

**Why this works**: The greedy choice at each level (max subsequence extraction, optimal merge) leads to the global optimum when we try all possible splits.

## Detailed Explanation
### Step 1: Max Subsequence from Single Array

Given array `nums` and target length `k`, find the maximum subsequence:

```python
def max_subsequence(nums, k):
    drop = len(nums) - k  # Number of elements we can drop
    stack = []
    
    for num in nums:
        # Pop smaller elements if we can afford to drop them
        while stack and stack[-1] < num and drop > 0:
            stack.pop()
            drop -= 1
        stack.append(num)
    
    return stack[:k]  # Take first k elements
```

**Key insight**: We maintain a monotonic decreasing stack. When we see a larger element, we can discard previous smaller elements (if we have drops left) to make room for the larger one.

**Example**: `nums = [9, 1, 2, 5, 8, 3], k = 3`
- Process 9: stack = [9], drop = 3
- Process 1: stack = [9, 1], drop = 3
- Process 2: stack = [9, 2], drop = 2 (popped 1)
- Process 5: stack = [9, 5], drop = 1 (popped 2)
- Process 8: stack = [9, 8], drop = 0 (popped 5)
- Process 3: stack = [9, 8, 3], drop = 0
- Result: [9, 8, 3]

### Step 2: Merge Two Sequences

Given two sequences, merge them to create the maximum result:

```python
def merge(nums1, nums2):
    result = []
    while nums1 or nums2:
        # Choose the sequence with larger remaining part
        if nums1 > nums2:  # Lexicographic comparison of entire sequences
            result.append(nums1[0])
            nums1 = nums1[1:]
        else:
            result.append(nums2[0])
            nums2 = nums2[1:]
    return result
```

**Critical detail**: The comparison `nums1 > nums2` compares entire sequences lexicographically, not just first elements. This handles cases like:
- `[6, 7]` vs `[6, 0, 4]` → choose from `[6, 7]` because `[7] > [0, 4]`

### Step 3: Try All Splits

```python
def max_number(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    result = [0] * k
    
    for i in range(max(0, k - n), min(k, m) + 1):
        # Take i from nums1, k-i from nums2
        sub1 = max_subsequence(nums1, i)
        sub2 = max_subsequence(nums2, k - i)
        merged = merge(sub1, sub2)
        result = max(result, merged)
    
    return result
```

**Range explanation**:
- Start: `max(0, k - n)` ensures we take at least `k - n` from nums1 if nums2 doesn't have enough
- End: `min(k, m)` ensures we don't take more than available from nums1

**Example walkthrough** for `nums1 = [3, 4, 6, 5], nums2 = [9, 1, 2, 5, 8, 3], k = 5`:
- i = 0: [] from nums1, [9,8,5,3,2] from nums2 → merge → [9,8,5,3,2]
- i = 1: [6] from nums1, [9,8,5,3] from nums2 → merge → [9,8,6,5,3]
- i = 2: [6,5] from nums1, [9,8,3] from nums2 → merge → [9,8,6,5,3]
- i = 3: [4,6,5] from nums1, [9,8] from nums2 → merge → [9,8,4,6,5]
- i = 4: [3,4,6,5] from nums1, [9] from nums2 → merge → [9,3,4,6,5]
- Maximum: [9,8,6,5,3]

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Greedy split + merge | O(k * (m + n)^2) | O(k) | Optimal solution. k splits, each O((m+n)^2) for merge. |
| Optimized merge | O(k * (m + n)) | O(k) | Use pointers instead of slicing. |
| Brute force | O(C(m+n,k) * k) | O(k) | Try all subsequences. Too slow. |
| DP approach | O(k^2 * m * n) | O(k * m * n) | More complex, not better. |

## Reference Implementation

In [ ]:
from typing import List


def max_number(nums1: List[int], nums2: List[int], k: int) -> List[int]:
    """
    Create maximum number of length k from two arrays.
    
    Args:
        nums1: First array of digits
        nums2: Second array of digits
        k: Length of result
    
    Returns:
        Maximum number as list of digits
    """
    def max_subsequence(nums: List[int], length: int) -> List[int]:
        """Extract maximum subsequence of given length from nums."""
        drop = len(nums) - length  # Number of elements we can drop
        stack = []
        
        for num in nums:
            # Pop smaller elements if we can afford to drop them
            while stack and stack[-1] < num and drop > 0:
                stack.pop()
                drop -= 1
            stack.append(num)
        
        return stack[:length]
    
    def merge(nums1: List[int], nums2: List[int]) -> List[int]:
        """Merge two sequences to create maximum result."""
        result = []
        while nums1 or nums2:
            # Compare entire remaining sequences lexicographically
            if nums1 > nums2:
                result.append(nums1[0])
                nums1 = nums1[1:]
            else:
                result.append(nums2[0])
                nums2 = nums2[1:]
        return result
    
    m, n = len(nums1), len(nums2)
    result = [0] * k
    
    # Try all possible splits
    for i in range(max(0, k - n), min(k, m) + 1):
        # Take i digits from nums1, k-i digits from nums2
        sub1 = max_subsequence(nums1, i)
        sub2 = max_subsequence(nums2, k - i)
        merged = merge(sub1, sub2)
        
        # Keep the maximum result
        if merged > result:
            result = merged
    
    return result

## Validation

In [ ]:
cases = [
    ([3, 4, 6, 5], [9, 1, 2, 5, 8, 3], 5, [9, 8, 6, 5, 3]),
    ([6, 7], [6, 0, 4], 5, [6, 7, 6, 0, 4]),
    ([3, 9], [8, 9], 3, [9, 8, 9]),
    ([2, 5, 6, 4, 4, 0], [7, 3, 8, 0, 6, 5, 7, 6, 2], 15, [7, 3, 8, 2, 5, 6, 4, 4, 0, 6, 5, 7, 6, 2, 0]),
    ([8, 9], [3, 9], 3, [9, 8, 9]),
    ([6, 7], [6, 0, 4], 3, [6, 7, 4]),
    ([3, 4, 6, 5], [9, 1, 2, 5, 8, 3], 3, [9, 8, 6]),
]

for nums1, nums2, k, expected in cases:
    result = max_number(nums1, nums2, k)
    assert result == expected, f"Failed for nums1={nums1}, nums2={nums2}, k={k}: got {result}, expected {expected}"

print('All tests passed for LC 321.')

## Complexity Analysis
- **Time Complexity**: O(k^2 * (m + n))
  - k possible splits: O(k)
  - Each split:
    - Extract subsequence from nums1: O(m)
    - Extract subsequence from nums2: O(n)
    - Merge: O(k * (m + n)) due to sequence comparisons and slicing
  - Total: O(k * (m + n + k * (m + n))) ≈ O(k^2 * (m + n))
  - Can be optimized to O(k * (m + n)) with pointer-based merge
- **Space Complexity**: O(k) for the result and intermediate subsequences
- **Bottleneck**: The merge operation with sequence comparisons

## Edge Cases & Pitfalls
- **k = m + n**: Must take all digits from both arrays.
- **k = 0**: Return empty array (though constraints say k >= 1).
- **One array empty**: Take all k digits from the other array.
- **Equal prefixes**: When merging [6,7] and [6,0,4], must compare suffixes [7] vs [0,4].
- **Comparison pitfall**: Don't just compare current elements; compare entire remaining sequences.
- **Stack overflow in recursion**: Use iterative approach for large inputs.
- **Slicing overhead**: In Python, list slicing creates copies; use indices for optimization.
- **Range calculation**: Ensure `max(0, k-n)` to `min(k, m)` covers all valid splits.

## Follow-up Variants
- Can you optimize the merge to O(k) time using pointers instead of slicing?
- What if digits can be rearranged within each array? (becomes a simpler sorting problem)
- How would you extend this to more than 2 arrays?
- What if you want the minimum number instead of maximum?
- Can you parallelize the split attempts for better performance?

## Takeaways
- Monotonic stack is powerful for finding maximum/minimum subsequences with order constraints.
- When merging sequences, lexicographic comparison of entire remainders is crucial.
- Breaking complex problems into subproblems (extract + merge) simplifies solutions.
- Trying all valid splits is acceptable when the range is limited.
- Greedy algorithms can be combined at multiple levels: greedy extraction + greedy merge.
- Pay attention to comparison semantics: element vs sequence comparison.
- Optimization: consider pointer-based approaches to avoid slicing overhead.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 402 | Remove K Digits | Greedy + Monotonic Stack |
| 316 | Remove Duplicate Letters | Greedy + Monotonic Stack |
| 1081 | Smallest Subsequence of Distinct Characters | Greedy + Monotonic Stack |
| 1673 | Find the Most Competitive Subsequence | Greedy + Monotonic Stack |
| 88 | Merge Sorted Array | Two Pointers |